# Flipkart Sentiment Analysis (YONEX MAVIS 350)

End-to-end NLP pipeline until model & vectorizer saving.

In [6]:

# Install & import libraries
import pandas as pd
import numpy as np
import re, string, pickle, nltk

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report

nltk.download('stopwords')
nltk.download('wordnet')

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [7]:
# Load dataset
import zipfile

zip_file_path = "/content/reviews_data_dump.zip"
csv_file_in_zip = "reviews_badminton/data.csv"

with zipfile.ZipFile(zip_file_path, 'r') as z:
    with z.open(csv_file_in_zip) as f:
        df = pd.read_csv(f)

df = df[['Review text', 'Ratings']]
df.rename(columns={'Review text':'review','Ratings':'rating'}, inplace=True)
df = df[df['rating'] != 3]
df['sentiment'] = df['rating'].apply(lambda x: 1 if x >= 4 else 0)

df.head()

,review,rating,sentiment
0,"Nice product, good quality, but price is now r...",4,1
1,They didn't supplied Yonex Mavis 350. Outside ...,1,0
2,Worst product. Damaged shuttlecocks packed in ...,1,0
4,Over pricedJust â?¹620 ..from retailer.I didn'...,1,0
5,Good quality product. Delivered on time.READ MORE,5,1


In [8]:

# Text Cleaning
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation))
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return " ".join(tokens)

df['clean_review'] = df['review'].astype(str).apply(clean_text)


In [9]:

# Train-test split
X = df['clean_review']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


In [10]:

# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)


In [11]:

# Model Training
model = LogisticRegression(max_iter=1000)
model.fit(X_train_vec, y_train)


LogisticRegression(max_iter=1000)

In [12]:

# Evaluation
y_pred = model.predict(X_test_vec)
print("F1 Score:", f1_score(y_test, y_pred))
print(classification_report(y_test, y_pred))


F1 Score: 0.9542715349166963
              precision    recall  f1-score   support

           0       0.84      0.49      0.62       215
           1       0.93      0.99      0.95      1366

    accuracy                           0.92      1581
   macro avg       0.88      0.74      0.79      1581
weighted avg       0.91      0.92      0.91      1581



In [13]:

# Save Model & Vectorizer
with open("sentiment_model.pkl","wb") as f:
    pickle.dump(model,f)

with open("tfidf_vectorizer.pkl","wb") as f:
    pickle.dump(tfidf,f)

print("Model and vectorizer saved")


Model and vectorizer saved
